# Diabetic Readmission Risk — v2

This notebook upgrades the v1 analysis in four ways:

1. **Proper encoding:** a scikit-learn `Pipeline` with one-hot encoding + scaling for Logistic Regression (v1 used `LabelEncoder`, which gives nominal features like `diag_1` a fake order).
2. **Gradient boosting:** LightGBM, default and tuned with 5-fold `RandomizedSearchCV`.
3. **Permutation importance:** checks which features *actually* move AUC, instead of relying only on impurity-based importance.
4. **Capacity-based thresholds:** "if the team can call the top N% riskiest patients, how many readmissions do we catch?"

Splits are identical to v1 (`random_state=42`, 70/15/15 stratified), so v1 and v2 numbers are directly comparable. The test set is used **once**, in the last section.

## 1. Setup and data

In [ ]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("dubradave/hospital-readmissions")
df = pd.read_csv(path + "/hospital_readmissions.csv")
print(df.shape)
df.head()

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
os.makedirs("figures", exist_ok=True)

# Set READMIT_CSV to use a local copy; otherwise download from Kaggle.
DATA_PATH = os.environ.get("READMIT_CSV")
if DATA_PATH is None:
    import kagglehub
    DATA_PATH = kagglehub.dataset_download("dubradave/hospital-readmissions") + "/hospital_readmissions.csv"

df = pd.read_csv(DATA_PATH)
df["readmitted"] = df["readmitted"].map({"no": 0, "yes": 1})
print(df.shape)
print(df["readmitted"].value_counts(normalize=True).round(4))

### Feature groups

- **Numeric (7):** utilisation and encounter counts → scaled for the linear model.
- **`age`:** genuinely ordered (10-year bands) → encoded as 0, 1, 2… in the right order.
- **Nominal (8):** specialty, diagnoses, test results, yes/no flags → one-hot for the linear model. `"Missing"` stays a category of its own.

In [ ]:
NUM_COLS = ["time_in_hospital", "n_lab_procedures", "n_procedures", "n_medications",
            "n_outpatient", "n_inpatient", "n_emergency"]
AGE_COL = ["age"]
NOM_COLS = ["medical_specialty", "diag_1", "diag_2", "diag_3",
            "glucose_test", "A1Ctest", "change", "diabetes_med"]
AGE_ORDER = sorted(df["age"].unique())   # "[40-50)" < "[50-60)" < ... < "[90-100)"
print("Age order:", AGE_ORDER)

X = df[NUM_COLS + AGE_COL + NOM_COLS]
y = df["readmitted"]

# Same split as v1: 15% test, then 15% of the total as validation
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=SEED)
print(X_train.shape, X_val.shape, X_test.shape)

## 2. Preprocessing pipelines

In [ ]:
# For Logistic Regression: scale numerics, ordered age, one-hot nominals
linear_prep = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("age", Pipeline([("ord", OrdinalEncoder(categories=[AGE_ORDER])), ("sc", StandardScaler())]), AGE_COL),
    ("nom", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), NOM_COLS),
])

# For tree models: integer codes are fine (trees split on thresholds, scale doesn't matter)
tree_prep = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("age", OrdinalEncoder(categories=[AGE_ORDER]), AGE_COL),
    ("nom", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), NOM_COLS),
])

# Exact v1 setup (alphabetical integer codes, no scaling) kept as a reference point
v1_prep = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), AGE_COL + NOM_COLS),
])

models = {
    "LogReg v1 (label-encoded)": Pipeline([("prep", v1_prep),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED))]),
    "LogReg v2 (one-hot + scaled)": Pipeline([("prep", linear_prep),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED))]),
    "LogReg v2 (balanced weights)": Pipeline([("prep", linear_prep),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))]),
    "Random Forest": Pipeline([("prep", tree_prep),
        ("model", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=1))]),
    "LightGBM (untuned)": Pipeline([("prep", tree_prep),
        ("model", LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                 random_state=SEED, verbose=-1, n_jobs=1))]),
}

## 3. Tune LightGBM

5-fold `RandomizedSearchCV` on train + validation (the test set is not touched). Only modest ranges: with 16 features, smaller trees and stronger regularisation usually win.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

param_dist = {
    "model__num_leaves": [7, 15, 31, 63],
    "model__min_child_samples": [20, 50, 100, 200],
    "model__learning_rate": [0.01, 0.03, 0.05],
    "model__n_estimators": [200, 400, 800],
    "model__reg_lambda": [0, 1, 5, 10],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
}
search = RandomizedSearchCV(
    Pipeline([("prep", tree_prep),
              ("model", LGBMClassifier(subsample=0.8, subsample_freq=1, random_state=SEED, verbose=-1, n_jobs=1))]),
    param_dist, n_iter=25, scoring="roc_auc", cv=cv, random_state=SEED, n_jobs=-1,
)
search.fit(X_temp, y_temp)
print("Best CV AUC:", round(search.best_score_, 4))
print("Best params:", {k.replace("model__", ""): v for k, v in search.best_params_.items()})

models["LightGBM (tuned)"] = clone(search.best_estimator_)

## 4. Compare all models

- **Validation metrics** (threshold 0.5), every model fit on the training set only.
- **5-fold CV** on train + validation, to show the spread across splits.

Note: the tuned LightGBM's parameters were picked using CV on data that includes the validation set, so its validation and CV numbers are slightly optimistic. The test set in Section 7 is the fair comparison.

In [ ]:
def metrics(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    return {"Accuracy": accuracy_score(y_true, pred), "Precision": precision_score(y_true, pred),
            "Recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred),
            "AUC": roc_auc_score(y_true, prob)}

val_rows, cv_rows, val_probs = {}, {}, {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    val_probs[name] = pipe.predict_proba(X_val)[:, 1]
    val_rows[name] = metrics(y_val, val_probs[name])
    res = cross_validate(clone(pipe), X_temp, y_temp, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_rows[name] = {"CV AUC mean": res["test_score"].mean(), "CV AUC std": res["test_score"].std()}

comparison = pd.DataFrame(val_rows).T.join(pd.DataFrame(cv_rows).T).round(4)
comparison = comparison.sort_values("CV AUC mean", ascending=False)
comparison

In [ ]:
# Sanity check: "LogReg v1" should reproduce v1's numbers (val AUC ≈ 0.6415, recall ≈ 0.40)
BEST = comparison.index[0]
print("Best model by CV AUC:", BEST)

fig, ax = plt.subplots(figsize=(8, 4))
order = comparison.index[::-1]
ax.barh(order, comparison.loc[order, "CV AUC mean"], xerr=comparison.loc[order, "CV AUC std"],
        color=["#2b6cb0" if n == BEST else "#a0aec0" for n in order])
ax.set_xlim(0.60, max(0.70, comparison["CV AUC mean"].max() + 0.01))
ax.set_xlabel("5-fold CV AUC (mean ± std)")
ax.set_title("Model comparison")
plt.tight_layout(); plt.savefig("figures/model_comparison.png", dpi=150); plt.show()

## 5. What actually drives the predictions?

Impurity-based importance (what v1 used) is biased toward numeric features with many distinct values, such as `n_lab_procedures`. **Permutation importance** shuffles one feature at a time on the validation set and measures how much AUC drops — a more honest measure.

In [ ]:
def perm_imp(pipe):
    r = permutation_importance(pipe, X_val, y_val, scoring="roc_auc", n_repeats=10,
                               random_state=SEED, n_jobs=-1)
    return pd.Series(r.importances_mean, index=X_val.columns)

rf_pipe = models["Random Forest"]
rf_impurity = pd.Series(rf_pipe.named_steps["model"].feature_importances_,
                        index=NUM_COLS + AGE_COL + NOM_COLS)
imp = pd.DataFrame({
    "RF impurity (v1 method)": rf_impurity,
    "RF permutation (ΔAUC)": perm_imp(rf_pipe),
    f"{BEST} permutation (ΔAUC)": perm_imp(models[BEST]),
}).sort_values(f"{BEST} permutation (ΔAUC)", ascending=False)
imp.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
top = imp.head(10)
axes[0].barh(top.index[::-1], top["RF impurity (v1 method)"][::-1], color="#a0aec0")
axes[0].set_title("Impurity importance (Random Forest)")
col = f"{BEST} permutation (ΔAUC)"
axes[1].barh(top.index[::-1], top[col][::-1], color="#2b6cb0")
axes[1].set_title(f"Permutation importance — {BEST}")
axes[1].set_xlabel("Drop in validation AUC when shuffled")
plt.tight_layout(); plt.savefig("figures/feature_importance.png", dpi=150); plt.show()

## 6. Capacity-based thresholds

A hospital doesn't think in thresholds — it thinks in "we can follow up with N patients a week". So instead of a probability cut-off, rank patients by risk and flag the top k%.

In [ ]:
def capacity_table(y_true, prob, shares=(0.10, 0.20, 0.30, 0.40, 0.50)):
    rows = []
    base = y_true.mean()
    for s in shares:
        thr = np.quantile(prob, 1 - s)
        pred = (prob >= thr).astype(int)
        p = precision_score(y_true, pred)
        rows.append({"Flag top": f"{int(s*100)}%", "Threshold": round(thr, 3),
                     "Precision": round(p, 3), "Recall": round(recall_score(y_true, pred), 3),
                     "Lift vs random": round(p / base, 2)})
    return pd.DataFrame(rows)

print(f"Validation — {BEST}")
cap_best = capacity_table(y_val, val_probs[BEST]); display(cap_best)
if BEST != "LogReg v2 (one-hot + scaled)":
    print("Validation — LogReg v2 (one-hot + scaled)")
    display(capacity_table(y_val, val_probs["LogReg v2 (one-hot + scaled)"]))

In [ ]:
# Pick an operating point on VALIDATION: highest-precision threshold that still catches ≥ 75% of readmissions
def threshold_for_recall(y_true, prob, target=0.75):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        pred = (prob >= t).astype(int)
        if recall_score(y_true, pred) >= target:
            best = t          # keep raising t while recall still meets the target
    return best

OP_THR = threshold_for_recall(y_val, val_probs[BEST], 0.75)
print(f"{BEST}: threshold for recall ≥ 0.75 on validation = {OP_THR:.3f}")

## 7. Final test-set evaluation (run once)

In [ ]:
test_rows = {}
for name, pipe in models.items():
    test_rows[name] = metrics(y_test, pipe.predict_proba(X_test)[:, 1])
test_df = pd.DataFrame(test_rows).T.round(4).loc[comparison.index]
display(test_df)

best_test_prob = models[BEST].predict_proba(X_test)[:, 1]
op = metrics(y_test, best_test_prob, OP_THR)
tn, fp, fn, tp = confusion_matrix(y_test, (best_test_prob >= OP_THR).astype(int)).ravel()
print(f"\n{BEST} at the validation-chosen threshold {OP_THR:.3f} (test set):")
print(f"  Recall {op['Recall']:.3f} | Precision {op['Precision']:.3f} | "
      f"flags {(tp+fp)/len(y_test):.0%} of patients | caught {tp} of {tp+fn} readmissions")

# Capacity view on the test set (pure ranking, nothing tuned here)
cap_test = capacity_table(y_test, best_test_prob)
print(f"\nCapacity-based screening on the test set — {BEST}")
display(cap_test)

## 8. README-ready tables

In [ ]:
def to_md(frame, digits=3, label="Model"):
    frame = frame.round(digits)
    head = f"| {label} | " + " | ".join(frame.columns) + " |"
    sep = "|---" * (len(frame.columns) + 1) + "|"
    body = ["| " + str(i) + " | " + " | ".join(str(v) for v in r) + " |" for i, r in frame.iterrows()]
    return "\n".join([head, sep] + body)

print("### Test set (threshold 0.5)\n")
print(to_md(test_df.join(comparison[["CV AUC mean", "CV AUC std"]])))
print(f"\n### Capacity-based screening (test set, {BEST})\n")
print(to_md(cap_test.set_index("Flag top"), label="Flag top"))
print("\n### Feature importance, top 8\n")
print(to_md(imp.head(8), 4, label="Feature"))